In [1]:
import datetime as dt

import luigi
import pandas as pd
from src.qr import get_mos_dates
from src.backtester import AuctionBacktester, OnTheRunBondsStrategy, DelegatingExecutor, MultCostModel, CostModel, \
    DailyDataLoader, HedgeTargetBuilder, IncrementalHedgeTargetBuilder
from calendar import Day
luigi.interface.core.log_level = "WARNING"

In [2]:
dates = get_mos_dates(dt.date(2025, 1, 1), dt.date(2026, 2, 18), incl_end=True)
backtester = AuctionBacktester(
    OnTheRunBondsStrategy(action_by_date={Day.WEDNESDAY: 'model_target'}, short_notionals=[500_000_000, 350_000_000, 150_000_000]),
    DelegatingExecutor(MultCostModel(CostModel(), 1.0)),
    DailyDataLoader(),
    HedgeTargetBuilder)
history = backtester.run(dates)
trades = pd.DataFrame(backtester.portfolio.trades)
trades['date'] = pd.to_datetime(trades['date'])

history['pnl'] = history['equity'].diff()
# history['unexplained_pnl'] = history['pnl'] - history[['holding_pnl', 'auction_exec_to_close_pnl', 'lob_exec_to_close_pnl']].sum(axis=1)

date = pd.Timestamp('20260211')
display(trades.query('date < @date').groupby('secid')[['size']].sum().query('abs(size) > 1'))
display(trades.query('date <= @date').groupby('secid')[['size']].sum().query('abs(size) > 1'))
display(trades.query('date == @date'))

history.filter(like='pnl').cumsum().plot(backend='plotly')

,size
secid,
SU26250RMFS9,-5.000000e+08
SU26253RMFS3,8.353555e+08
SU26254RMFS1,-3.500000e+08


,size
secid,
SU26249RMFS1,1.346940e+09
SU26251RMFS7,-1.500000e+08
SU26253RMFS3,-3.500000e+08
SU26254RMFS1,-5.000000e+08


,date,secid,size,price,value,source
198,2026-02-11,SU26251RMFS7,-1.500000e+08,84.109715,-1.261646e+08,lob
199,2026-02-11,SU26249RMFS1,1.346940e+09,85.020000,1.145168e+09,auction
200,2026-02-11,SU26254RMFS1,-1.500000e+08,90.016386,-1.350246e+08,lob
201,2026-02-11,SU26250RMFS9,5.000000e+08,84.900000,4.245000e+08,auction
202,2026-02-11,SU26253RMFS3,-1.185356e+09,90.238042,-1.069642e+09,lob
